<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/pytochCNNPractice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os
from google.colab import userdata
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets,transforms
from torch.utils.data import DataLoader

torch.backends.cudnn.benchmark = True

In [4]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('username and key activated')

username and key activated


In [5]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:51<00:00, 78.1MB/s]



In [7]:
!unzip -q 140k-real-and-fake-faces.zip -d ./dataset_folder

In [8]:
train_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/train'
test_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/test'
validation_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/valid'

In [15]:
# =========================
# 1. DEVICE (CPU / GPU)
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [17]:
# =========================
# 2. TRANSFORMS
# =========================

train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

validation_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

In [19]:
# =========================
# 3. DATASET + DATALOADER
# =========================
train_data = datasets.ImageFolder(root=train_dir,transform=train_transform)
validation_data = datasets.ImageFolder(root=validation_dir,transform=validation_transform)

train_dataload = DataLoader(dataset=train_data,batch_size=32,shuffle=True,num_workers=2,pin_memory=True)
validation_dataload = DataLoader(dataset=validation_data,batch_size=32,num_workers=2,pin_memory=True)

In [20]:
# =========================
# 4. MODEL (CNN)
# =========================

class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv = nn.Sequential(
        nn.Conv2d(3,32,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(32,32,3),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(64,64,3),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
    )
    self.flatten = nn.Flatten()

    self.fc = nn.Sequential(
        nn.LazyLinear(128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU()
    )

  def forward(self,x):
    x = self.conv(x)
    x = self.flatten(x)
    x = self.fc(x)
    return x

In [21]:
model = CNN().to(device)

In [23]:
# =========================
# 5. LOSS + OPTIMIZER
# =========================
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(params=model.parameters(),lr=0.001)

In [ ]:
# =========================
# 6. TRAINING LOOP
# =========================
epochs = 5

for epoch in range(epochs):
  model.train()

  train_loss = 0
  train_correct = 0
  train_total = 0

  for images,labels in train_dataload:
    images = images.to(device)
    labels = labels.float().unsqueeze(1).to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = loss_fn(outputs,labels)

    loss.backward()
    optimizer.step() # here it goes for checking weights with learning rate and gradients and apply.

    train_loss += loss.item()

    preds = (torch.sigmoid(outputs) > 0.5).int()
    train_correct += (preds.squeeze() == labels.squeeze()).sum().item()
    train_total = labels.size(0)

  train_accuracy = train_correct/train_total

  # -------- VALIDATION --------
  model.eval()

  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for images,labels in validation_dataload:
      images = images.to(device)
      labels = labels.float().unsqueeze(1).to(device)

      outputs = model(images)
      loss = loss_fn(outputs,labels)

      val_loss += loss.item() # item basically convert value from tensor to normal
      preds = (torch.sigmoid(outputs) > 0.5).int()
      val_correct += (preds.squeeze() == labels.squeeze()).sum().item()
      val_total = labels.size(0)

  val_accuracy = val_correct / val_total

